#1\. Initialization

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab

Mounted at /content/drive
/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab


In [2]:
SEED = 123
EPOCHS = 20

In [3]:
import os
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be before torch import

In [4]:
import math, random, hashlib, copy
import pandas as pd
import numpy as np

from pathlib import Path
from typing  import Tuple, List
from PIL     import Image, ImageEnhance

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [5]:
print("Device:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("PyTorch version:", torch.__version__)

Device: Tesla T4
CUDA version: 12.8
cuDNN version: 91900
PyTorch version: 2.11.0+cu128


In [6]:
sc4b_loss_function_result = []
sc4b_reliability_result   = []

#2\. Data Loading

In [7]:
val_sc4b_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4b/val_sc4b.xlsx")
test_sc4b_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4b/test_sc4b.xlsx")

#3\. Pre-processing Set

In [8]:
# Transformation

IMAGE_SIZE = 224

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

In [9]:
# Dataset Loader

BATCH_SIZE = 32
NUM_WORKERS = 2

class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx]
        image  = Image.open(sample["path"]).convert("RGB")
        image  = self.transform(image) if self.transform else transforms.ToTensor()(image)
        label  = int(sample["label"])
        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "child_id": str(sample["child_id"]),
            "path": str(sample["path"]),
        }

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(dataframe, transform, shuffle):
    ds = FaceDataset(dataframe, transform=transform)
    g  = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=g,
        worker_init_fn=seed_worker,  # ← ADD THIS
    )

#4\. Sc.4.b. EfficientNet-B0 with SGD

In [10]:
# Prediction: Function

def collect_predictions(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            xb = batch["image"].to(device, non_blocking=True)
            yb = batch["label"].cpu().numpy().astype(int)
            logits = model(xb)
            probs  = torch.softmax(logits, dim=1).detach().cpu().numpy()
            logits_np = logits.detach().cpu().numpy()

            for i in range(len(yb)):
                rows.append({
                    "child_id": batch["child_id"][i],
                    "path": batch["path"][i],
                    "label": int(yb[i]),
                    "logit0": float(logits_np[i, 0]),
                    "logit1": float(logits_np[i, 1]),
                    "prob0": float(probs[i, 0]),
                    "prob1": float(probs[i, 1])
                })
    return pd.DataFrame(rows)

In [11]:
#Evaluation Function

def safe_div(num, den):
    return float(num) / float(den) if den else 0.0

def metric_bundle(y_true, prob1, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob1 = np.asarray(prob1).astype(float)
    y_pred = (prob1 >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy = safe_div(tn + tp, len(y_true))
    precision = safe_div(tp, tp + fp)
    sensitivity = safe_div(tp, tp + fn)   # recall for stunting
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * sensitivity, precision + sensitivity)
    beta2 = 2.0
    f2 = safe_div((1 + beta2**2) * precision * sensitivity, (beta2**2) * precision + sensitivity)
    bal_acc = 0.5 * (sensitivity + specificity)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
        "f2": f2,
        "balanced_accuracy": bal_acc,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


In [12]:
# Bootstrap Resample

def bootstrap_subject_ci(subject_df, prob_col, threshold, n_boot=2000, seed=123):
    rng  = np.random.default_rng(seed)
    base = subject_df[["child_id", "label", prob_col]].copy().reset_index(drop=True)
    n    = len(base)

    metrics = {
        "accuracy": [], "precision": [], "sensitivity": [],
        "specificity": [], "f1": [], "f2": [], "balanced_accuracy": []
    }

    for _ in range(n_boot):
        idx    = rng.integers(0, n, size=n)
        sample = base.iloc[idx]
        m      = metric_bundle(sample["label"].values, sample[prob_col].values, threshold)
        for k in metrics:
            metrics[k].append(m[k])

    ci = {}
    for k, vals in metrics.items():
        ci[f"{k}_mean"]     = float(np.round(np.mean(vals)*100, 2))
        ci[f"{k}_std. dev"] = float(np.round(np.std(vals)*100, 2))
        ci[f"{k}_ci_low"]   = float(np.round(np.percentile(vals, 2.5)*100, 2))
        ci[f"{k}_ci_high"]  = float(np.round(np.percentile(vals, 97.5)*100, 2))
    return ci

In [13]:
def threshold_sweep(subject_df, prob_col, thresholds, objective="f2"):
    rows = []
    for thr in thresholds:
        m = metric_bundle(subject_df["label"].values, subject_df[prob_col].values, threshold=float(thr))
        rows.append({"threshold": float(thr), **m})

    table = pd.DataFrame(rows).sort_values(["threshold"]).reset_index(drop=True)
    best  = table.sort_values(
        by=[objective, "sensitivity", "specificity"],
        ascending=[False, False, False]
    ).iloc[0]
    return table, float(best["threshold"])


In [14]:
def fit_temperature(logits_np, labels_np, max_iter=50):
    logits_t = torch.tensor(logits_np, dtype=torch.float32)
    labels_t = torch.tensor(labels_np, dtype=torch.long)

    log_temp = nn.Parameter(torch.zeros(()))  # T = exp(log_temp), starts at 1.0
    optimizer = torch.optim.LBFGS([log_temp], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        temp = torch.exp(log_temp).clamp(1e-3, 100.0)
        loss = F.cross_entropy(logits_t / temp, labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.exp(log_temp).detach().cpu().item())

def apply_temperature(pred_df, temperature):
    logits   = pred_df[["logit0", "logit1"]].to_numpy(dtype=np.float32)
    logits_t = torch.tensor(logits, dtype=torch.float32)
    probs    = torch.softmax(logits_t / float(temperature), dim=1)[:, 1].numpy()
    out = pred_df.copy()
    out["prob1_cal"] = probs
    return out

In [15]:
# Model - pre-trained EfficientNet B0

def create_model(num_classes=2):
    weights = EfficientNet_B0_Weights.DEFAULT
    model   = efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

## 4\.1\. CE Loss Function

### 4\.1\.1\. Model Initialization

In [16]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 126MB/s] 


### 4\.1\.2\. Data Loader

In [17]:
# Load Dataset
dl_val_sc4b   = make_loader(val_sc4b_df, eval_tfms, shuffle=False)
dl_test_sc4b  = make_loader(test_sc4b_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc4b_df)} | test={len(test_sc4b_df)}")


val=19 | test=19


### 4\.1\.3\. Model Evaluation

In [18]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4b_sgd_CE.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc4b)
test_predict  = collect_predictions(model, dl_test_sc4b)

### 4\.1\.4\. Validation Data Performance

In [19]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.5789473684210527,
 'precision': 0.5,
 'sensitivity': 0.5,
 'specificity': 0.6363636363636364,
 'f1': 0.5,
 'f2': 0.5,
 'balanced_accuracy': 0.5681818181818181,
 'TN': 7,
 'FP': 4,
 'FN': 4,
 'TP': 4}

In [20]:
sc4b_loss_function_result.append([
     "sc4b-efficientnet-b0-sgd",
     "sc4b", "efficientnet-b0", "sgd", "CE",
     "validation",
     result["accuracy"],
     result["precision"],
     result["sensitivity"],
     result["specificity"],
     result["f1"],
     result["f2"],
     result["TN"],
     result["FP"],
     result["FN"],
     result["TP"],
])

### 4\.1\.5\. Test Data Performance

In [21]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.5263157894736842,
 'precision': 0.45454545454545453,
 'sensitivity': 0.625,
 'specificity': 0.45454545454545453,
 'f1': 0.5263157894736842,
 'f2': 0.5813953488372092,
 'balanced_accuracy': 0.5397727272727273,
 'TN': 5,
 'FP': 6,
 'FN': 3,
 'TP': 5}

In [22]:
sc4b_loss_function_result.append([
     "sc4b-efficientnet-b0-sgd",
     "sc4b", "efficientnet-b0", "sgd", "CE",
     "test",
     result["accuracy"],
     result["precision"],
     result["sensitivity"],
     result["specificity"],
     result["f1"],
     result["f2"],
     result["TN"],
     result["FP"],
     result["FN"],
     result["TP"],
])

### 4\.1\.6\. Static Threshold

In [23]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=0.5, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 52.67,
 'accuracy_std. dev': 11.52,
 'accuracy_ci_low': 31.58,
 'accuracy_ci_high': 73.68,
 'precision_mean': 45.59,
 'precision_std. dev': 15.65,
 'precision_ci_low': 15.38,
 'precision_ci_high': 75.0,
 'sensitivity_mean': 62.38,
 'sensitivity_std. dev': 18.31,
 'sensitivity_ci_low': 25.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 45.69,
 'specificity_std. dev': 15.15,
 'specificity_ci_low': 15.38,
 'specificity_ci_high': 75.0,
 'f1_mean': 51.29,
 'f1_std. dev': 14.61,
 'f1_ci_low': 21.03,
 'f1_ci_high': 76.19,
 'f2_mean': 56.84,
 'f2_std. dev': 15.59,
 'f2_ci_low': 23.26,
 'f2_ci_high': 83.33,
 'balanced_accuracy_mean': 54.03,
 'balanced_accuracy_std. dev': 11.89,
 'balanced_accuracy_ci_low': 31.11,
 'balanced_accuracy_ci_high': 76.19}

In [24]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "CE", "0.5", 0.5,
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

### 4\.1\.7\. Tuned Threshold

In [25]:
# Raw subject-level probabilities

THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 181), 3)
THRESHOLD_OBJECTIVE = "f2"

raw_curve, raw_thr = threshold_sweep(val_predict, prob_col="prob1",
                                     thresholds=THRESHOLD_GRID,
                                     objective=THRESHOLD_OBJECTIVE)

print(f"Best Threshold on Validation: {raw_thr:0.4f}")
display(raw_curve[:5])

Best Threshold on Validation: 0.0500


,threshold,accuracy,precision,sensitivity,specificity,f1,f2,balanced_accuracy,TN,FP,FN,TP
0,0.050,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
1,0.055,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
2,0.060,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
3,0.065,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
4,0.070,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8


In [26]:
# Evaluation Example
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=raw_thr)
display(raw_thr)
display(result)

test evaluation:


0.05

{'accuracy': 0.42105263157894735,
 'precision': 0.42105263157894735,
 'sensitivity': 1.0,
 'specificity': 0.0,
 'f1': 0.5925925925925926,
 'f2': 0.7843137254901961,
 'balanced_accuracy': 0.5,
 'TN': 0,
 'FP': 11,
 'FN': 0,
 'TP': 8}

In [27]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=raw_thr, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 42.27,
 'accuracy_std. dev': 11.42,
 'accuracy_ci_low': 21.05,
 'accuracy_ci_high': 63.16,
 'precision_mean': 42.27,
 'precision_std. dev': 11.42,
 'precision_ci_low': 21.05,
 'precision_ci_high': 63.16,
 'sensitivity_mean': 100.0,
 'sensitivity_std. dev': 0.0,
 'sensitivity_ci_low': 100.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 0.0,
 'specificity_std. dev': 0.0,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 0.0,
 'f1_mean': 58.5,
 'f1_std. dev': 11.48,
 'f1_ci_low': 34.78,
 'f1_ci_high': 77.42,
 'f2_mean': 77.11,
 'f2_std. dev': 8.73,
 'f2_ci_low': 57.14,
 'f2_ci_high': 89.55,
 'balanced_accuracy_mean': 50.0,
 'balanced_accuracy_std. dev': 0.0,
 'balanced_accuracy_ci_low': 50.0,
 'balanced_accuracy_ci_high': 50.0}

In [28]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "CE", "tuned", np.round(raw_thr, 3),
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

### 4\.1\.8\. Calibrated Tuned Threshold

In [29]:
# Temperature scaling on validation logits -> calibrated probs
temperature = fit_temperature(
    val_predict[["logit0", "logit1"]].to_numpy(dtype=np.float32),
    val_predict["label"].to_numpy(dtype=np.int64),
    max_iter=50,
    )

print('temperature:', temperature)

val_pred_cal = apply_temperature(val_predict, temperature)
val_pred_cal.head(5)

temperature: 103.59519958496094


,child_id,path,label,logit0,logit1,prob0,prob1,prob1_cal
0,4,/content/drive/MyDrive/Stunted Children Identi...,1,-0.109063,-0.105324,0.499065,0.500935,0.500009
1,13,/content/drive/MyDrive/Stunted Children Identi...,0,-0.099534,-0.178914,0.519835,0.480165,0.499808
2,24,/content/drive/MyDrive/Stunted Children Identi...,0,0.093452,-0.159351,0.562866,0.437134,0.499390
3,28,/content/drive/MyDrive/Stunted Children Identi...,0,-0.053447,0.157443,0.447472,0.552528,0.500509
4,37,/content/drive/MyDrive/Stunted Children Identi...,0,0.050591,-0.170715,0.555102,0.444898,0.499466


In [30]:
# Raw subject-level probabilities

THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 181), 3)
THRESHOLD_OBJECTIVE = "f2"

cal_curve, cal_thr = threshold_sweep(val_pred_cal, prob_col="prob1",
                                     thresholds=THRESHOLD_GRID, objective=THRESHOLD_OBJECTIVE)

print(f"Best Threshold on Validation: {cal_thr:0.4f}")
display(cal_curve[:5])

Best Threshold on Validation: 0.0500


,threshold,accuracy,precision,sensitivity,specificity,f1,f2,balanced_accuracy,TN,FP,FN,TP
0,0.050,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
1,0.055,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
2,0.060,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
3,0.065,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
4,0.070,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8


In [31]:
print('test evaluation:')
# Evaluation Example
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=cal_thr)
display(result)

test evaluation:


{'accuracy': 0.42105263157894735,
 'precision': 0.42105263157894735,
 'sensitivity': 1.0,
 'specificity': 0.0,
 'f1': 0.5925925925925926,
 'f2': 0.7843137254901961,
 'balanced_accuracy': 0.5,
 'TN': 0,
 'FP': 11,
 'FN': 0,
 'TP': 8}

In [32]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=cal_thr, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 42.27,
 'accuracy_std. dev': 11.42,
 'accuracy_ci_low': 21.05,
 'accuracy_ci_high': 63.16,
 'precision_mean': 42.27,
 'precision_std. dev': 11.42,
 'precision_ci_low': 21.05,
 'precision_ci_high': 63.16,
 'sensitivity_mean': 100.0,
 'sensitivity_std. dev': 0.0,
 'sensitivity_ci_low': 100.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 0.0,
 'specificity_std. dev': 0.0,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 0.0,
 'f1_mean': 58.5,
 'f1_std. dev': 11.48,
 'f1_ci_low': 34.78,
 'f1_ci_high': 77.42,
 'f2_mean': 77.11,
 'f2_std. dev': 8.73,
 'f2_ci_low': 57.14,
 'f2_ci_high': 89.55,
 'balanced_accuracy_mean': 50.0,
 'balanced_accuracy_std. dev': 0.0,
 'balanced_accuracy_ci_low': 50.0,
 'balanced_accuracy_ci_high': 50.0}

In [33]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "CE", "cal. tuned", np.round(cal_thr, 3),
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

## 4\.2\. Class-Weighted CE Loss Function

### 4\.2\.1\. Model Initialization

In [34]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


### 4\.2\.2\. Data Loader

In [35]:
# Load Dataset
dl_val_sc4b   = make_loader(val_sc4b_df, eval_tfms, shuffle=False)
dl_test_sc4b  = make_loader(test_sc4b_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc4b_df)} | test={len(test_sc4b_df)}")


val=19 | test=19


### 4\.2\.3\. Model Evaluation

In [36]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4b_sgd_weighted_CE.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc4b)
test_predict  = collect_predictions(model, dl_test_sc4b)

### 4\.2\.4\. Validation Data Performance

In [37]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.5263157894736842,
 'precision': 0.45454545454545453,
 'sensitivity': 0.625,
 'specificity': 0.45454545454545453,
 'f1': 0.5263157894736842,
 'f2': 0.5813953488372092,
 'balanced_accuracy': 0.5397727272727273,
 'TN': 5,
 'FP': 6,
 'FN': 3,
 'TP': 5}

In [38]:
sc4b_loss_function_result.append([
     "sc4b-efficientnet-b0-sgd",
     "sc4b", "efficientnet-b0", "sgd", "weighted-CE",
     "validation",
     result["accuracy"],
     result["precision"],
     result["sensitivity"],
     result["specificity"],
     result["f1"],
     result["f2"],
     result["TN"],
     result["FP"],
     result["FN"],
     result["TP"],
])

### 4\.2\.5\. Test Data Performance

In [39]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.5263157894736842,
 'precision': 0.46153846153846156,
 'sensitivity': 0.75,
 'specificity': 0.36363636363636365,
 'f1': 0.5714285714285714,
 'f2': 0.6666666666666666,
 'balanced_accuracy': 0.5568181818181819,
 'TN': 4,
 'FP': 7,
 'FN': 2,
 'TP': 6}

In [40]:
sc4b_loss_function_result.append([
     "sc4b-efficientnet-b0-sgd",
     "sc4b", "efficientnet-b0", "sgd", "weighted-CE",
     "test",
     result["accuracy"],
     result["precision"],
     result["sensitivity"],
     result["specificity"],
     result["f1"],
     result["f2"],
     result["TN"],
     result["FP"],
     result["FN"],
     result["TP"],
])

### 4\.2\.6\. Static Threshold

In [41]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=0.5, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 52.66,
 'accuracy_std. dev': 11.26,
 'accuracy_ci_low': 31.58,
 'accuracy_ci_high': 73.68,
 'precision_mean': 46.27,
 'precision_std. dev': 14.03,
 'precision_ci_low': 20.0,
 'precision_ci_high': 72.74,
 'sensitivity_mean': 74.93,
 'sensitivity_std. dev': 16.05,
 'sensitivity_ci_low': 40.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 36.38,
 'specificity_std. dev': 14.46,
 'specificity_ci_low': 9.09,
 'specificity_ci_high': 66.67,
 'f1_mean': 56.0,
 'f1_std. dev': 13.31,
 'f1_ci_low': 26.67,
 'f1_ci_high': 78.57,
 'f2_mean': 65.37,
 'f2_std. dev': 13.58,
 'f2_ci_low': 33.33,
 'f2_ci_high': 87.5,
 'balanced_accuracy_mean': 55.66,
 'balanced_accuracy_std. dev': 10.67,
 'balanced_accuracy_ci_low': 34.09,
 'balanced_accuracy_ci_high': 75.73}

In [42]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "Weighted CE", "0.5", 0.5,
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

### 4\.2\.7\. Tuned Threshold

In [43]:
# Raw subject-level probabilities

THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 181), 3)
THRESHOLD_OBJECTIVE = "f2"

raw_curve, raw_thr = threshold_sweep(val_predict, prob_col="prob1",
                                     thresholds=THRESHOLD_GRID,
                                     objective=THRESHOLD_OBJECTIVE)

print(f"Best Threshold on Validation: {raw_thr:0.4f}")
display(raw_curve[:5])

Best Threshold on Validation: 0.0500


,threshold,accuracy,precision,sensitivity,specificity,f1,f2,balanced_accuracy,TN,FP,FN,TP
0,0.050,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
1,0.055,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
2,0.060,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
3,0.065,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
4,0.070,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8


In [44]:
# Evaluation Example
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=raw_thr)
display(raw_thr)
display(result)

test evaluation:


0.05

{'accuracy': 0.42105263157894735,
 'precision': 0.42105263157894735,
 'sensitivity': 1.0,
 'specificity': 0.0,
 'f1': 0.5925925925925926,
 'f2': 0.7843137254901961,
 'balanced_accuracy': 0.5,
 'TN': 0,
 'FP': 11,
 'FN': 0,
 'TP': 8}

In [45]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=raw_thr, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 42.27,
 'accuracy_std. dev': 11.42,
 'accuracy_ci_low': 21.05,
 'accuracy_ci_high': 63.16,
 'precision_mean': 42.27,
 'precision_std. dev': 11.42,
 'precision_ci_low': 21.05,
 'precision_ci_high': 63.16,
 'sensitivity_mean': 100.0,
 'sensitivity_std. dev': 0.0,
 'sensitivity_ci_low': 100.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 0.0,
 'specificity_std. dev': 0.0,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 0.0,
 'f1_mean': 58.5,
 'f1_std. dev': 11.48,
 'f1_ci_low': 34.78,
 'f1_ci_high': 77.42,
 'f2_mean': 77.11,
 'f2_std. dev': 8.73,
 'f2_ci_low': 57.14,
 'f2_ci_high': 89.55,
 'balanced_accuracy_mean': 50.0,
 'balanced_accuracy_std. dev': 0.0,
 'balanced_accuracy_ci_low': 50.0,
 'balanced_accuracy_ci_high': 50.0}

In [46]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "Weighted CE", "tuned", np.round(raw_thr, 3),
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

### 4\.2\.8\. Calibrated Tuned Threshold

In [47]:
# Temperature scaling on validation logits -> calibrated probs
temperature = fit_temperature(
    val_predict[["logit0", "logit1"]].to_numpy(dtype=np.float32),
    val_predict["label"].to_numpy(dtype=np.int64),
    max_iter=50,
    )

print('temperature:', temperature)

val_pred_cal = apply_temperature(val_predict, temperature)
val_pred_cal.head(5)

temperature: 106.65105438232422


,child_id,path,label,logit0,logit1,prob0,prob1,prob1_cal
0,4,/content/drive/MyDrive/Stunted Children Identi...,1,-0.133999,-0.080130,0.486536,0.513464,0.500126
1,13,/content/drive/MyDrive/Stunted Children Identi...,0,-0.112495,-0.165963,0.513364,0.486636,0.499875
2,24,/content/drive/MyDrive/Stunted Children Identi...,0,0.073477,-0.139298,0.552994,0.447006,0.499501
3,28,/content/drive/MyDrive/Stunted Children Identi...,0,-0.091918,0.196065,0.428498,0.571502,0.500675
4,37,/content/drive/MyDrive/Stunted Children Identi...,0,0.026106,-0.145988,0.542918,0.457082,0.499597


In [48]:
# Raw subject-level probabilities

THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 181), 3)
THRESHOLD_OBJECTIVE = "f2"

cal_curve, cal_thr = threshold_sweep(val_pred_cal, prob_col="prob1",
                                     thresholds=THRESHOLD_GRID, objective=THRESHOLD_OBJECTIVE)

print(f"Best Threshold on Validation: {cal_thr:0.4f}")
display(cal_curve[:5])

Best Threshold on Validation: 0.0500


,threshold,accuracy,precision,sensitivity,specificity,f1,f2,balanced_accuracy,TN,FP,FN,TP
0,0.050,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
1,0.055,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
2,0.060,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
3,0.065,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
4,0.070,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8


In [49]:
print('test evaluation:')
# Evaluation Example
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=cal_thr)
display(result)

test evaluation:


{'accuracy': 0.42105263157894735,
 'precision': 0.42105263157894735,
 'sensitivity': 1.0,
 'specificity': 0.0,
 'f1': 0.5925925925925926,
 'f2': 0.7843137254901961,
 'balanced_accuracy': 0.5,
 'TN': 0,
 'FP': 11,
 'FN': 0,
 'TP': 8}

In [50]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=cal_thr, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 42.27,
 'accuracy_std. dev': 11.42,
 'accuracy_ci_low': 21.05,
 'accuracy_ci_high': 63.16,
 'precision_mean': 42.27,
 'precision_std. dev': 11.42,
 'precision_ci_low': 21.05,
 'precision_ci_high': 63.16,
 'sensitivity_mean': 100.0,
 'sensitivity_std. dev': 0.0,
 'sensitivity_ci_low': 100.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 0.0,
 'specificity_std. dev': 0.0,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 0.0,
 'f1_mean': 58.5,
 'f1_std. dev': 11.48,
 'f1_ci_low': 34.78,
 'f1_ci_high': 77.42,
 'f2_mean': 77.11,
 'f2_std. dev': 8.73,
 'f2_ci_low': 57.14,
 'f2_ci_high': 89.55,
 'balanced_accuracy_mean': 50.0,
 'balanced_accuracy_std. dev': 0.0,
 'balanced_accuracy_ci_low': 50.0,
 'balanced_accuracy_ci_high': 50.0}

In [51]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "Weighted CE", "cal. tuned", np.round(cal_thr, 3),
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

## 4\.3\. Focal Loss Function

### 4\.3\.1\. Model Initialization

In [52]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


### 4\.3\.2\. Data Loader

In [53]:
# Load Dataset
dl_val_sc4b   = make_loader(val_sc4b_df, eval_tfms, shuffle=False)
dl_test_sc4b  = make_loader(test_sc4b_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc4b_df)} | test={len(test_sc4b_df)}")


val=19 | test=19


### 4\.3\.3\. Model Evaluation

In [54]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4b_sgd_focal.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc4b)
test_predict  = collect_predictions(model, dl_test_sc4b)

### 4\.3\.4\. Validation Data Performance

In [55]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.5263157894736842,
 'precision': 0.46153846153846156,
 'sensitivity': 0.75,
 'specificity': 0.36363636363636365,
 'f1': 0.5714285714285714,
 'f2': 0.6666666666666666,
 'balanced_accuracy': 0.5568181818181819,
 'TN': 4,
 'FP': 7,
 'FN': 2,
 'TP': 6}

In [56]:
sc4b_loss_function_result.append([
    "sc4b-efficientnet-b0-sgd",
    "sc4b", "efficientnet-b0", "sgd", "focal",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

### 4\.3\.5\. Test Data Performance

In [57]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.3684210526315789,
 'precision': 0.375,
 'sensitivity': 0.75,
 'specificity': 0.09090909090909091,
 'f1': 0.5,
 'f2': 0.625,
 'balanced_accuracy': 0.42045454545454547,
 'TN': 1,
 'FP': 10,
 'FN': 2,
 'TP': 6}

In [58]:
sc4b_loss_function_result.append([
     "sc4b-efficientnet-b0-sgd",
     "sc4b", "efficientnet-b0", "sgd", "focal",
     "test",
     result["accuracy"],
     result["precision"],
     result["sensitivity"],
     result["specificity"],
     result["f1"],
     result["f2"],
     result["TN"],
     result["FP"],
     result["FN"],
     result["TP"],
])

### 4\.3\.6\. Static Threshold

In [59]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=0.5, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 36.83,
 'accuracy_std. dev': 10.73,
 'accuracy_ci_low': 15.79,
 'accuracy_ci_high': 57.89,
 'precision_mean': 37.58,
 'precision_std. dev': 11.99,
 'precision_ci_low': 15.79,
 'precision_ci_high': 61.54,
 'sensitivity_mean': 74.69,
 'sensitivity_std. dev': 15.41,
 'sensitivity_ci_low': 40.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 9.15,
 'specificity_std. dev': 8.81,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 30.0,
 'f1_mean': 48.93,
 'f1_std. dev': 12.31,
 'f1_ci_low': 22.22,
 'f1_ci_high': 71.43,
 'f2_mean': 60.97,
 'f2_std. dev': 12.71,
 'f2_ci_low': 32.61,
 'f2_ci_high': 81.97,
 'balanced_accuracy_mean': 41.92,
 'balanced_accuracy_std. dev': 8.78,
 'balanced_accuracy_ci_low': 25.0,
 'balanced_accuracy_ci_high': 58.33}

In [60]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "Focal", "0.5", 0.5,
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

### 4\.3\.7\. Tuned Threshold

In [61]:
# Raw subject-level probabilities

THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 181), 3)
THRESHOLD_OBJECTIVE = "f2"

raw_curve, raw_thr = threshold_sweep(val_predict, prob_col="prob1",
                                     thresholds=THRESHOLD_GRID,
                                     objective=THRESHOLD_OBJECTIVE)

print(f"Best Threshold on Validation: {raw_thr:0.4f}")
display(raw_curve[:5])

Best Threshold on Validation: 0.0500


,threshold,accuracy,precision,sensitivity,specificity,f1,f2,balanced_accuracy,TN,FP,FN,TP
0,0.050,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
1,0.055,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
2,0.060,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
3,0.065,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
4,0.070,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8


In [62]:
# Evaluation Example
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=raw_thr)
display(raw_thr)
display(result)

test evaluation:


0.05

{'accuracy': 0.42105263157894735,
 'precision': 0.42105263157894735,
 'sensitivity': 1.0,
 'specificity': 0.0,
 'f1': 0.5925925925925926,
 'f2': 0.7843137254901961,
 'balanced_accuracy': 0.5,
 'TN': 0,
 'FP': 11,
 'FN': 0,
 'TP': 8}

In [63]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=raw_thr, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 42.27,
 'accuracy_std. dev': 11.42,
 'accuracy_ci_low': 21.05,
 'accuracy_ci_high': 63.16,
 'precision_mean': 42.27,
 'precision_std. dev': 11.42,
 'precision_ci_low': 21.05,
 'precision_ci_high': 63.16,
 'sensitivity_mean': 100.0,
 'sensitivity_std. dev': 0.0,
 'sensitivity_ci_low': 100.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 0.0,
 'specificity_std. dev': 0.0,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 0.0,
 'f1_mean': 58.5,
 'f1_std. dev': 11.48,
 'f1_ci_low': 34.78,
 'f1_ci_high': 77.42,
 'f2_mean': 77.11,
 'f2_std. dev': 8.73,
 'f2_ci_low': 57.14,
 'f2_ci_high': 89.55,
 'balanced_accuracy_mean': 50.0,
 'balanced_accuracy_std. dev': 0.0,
 'balanced_accuracy_ci_low': 50.0,
 'balanced_accuracy_ci_high': 50.0}

In [64]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "Focal", "tuned", np.round(raw_thr, 3),
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

### 4\.3\.8\. Calibrated Tuned Threshold

In [65]:
# Temperature scaling on validation logits -> calibrated probs
temperature = fit_temperature(
    val_predict[["logit0", "logit1"]].to_numpy(dtype=np.float32),
    val_predict["label"].to_numpy(dtype=np.int64),
    max_iter=50,
    )

print('temperature:', temperature)

val_pred_cal = apply_temperature(val_predict, temperature)
val_pred_cal.head(5)

temperature: 126.0498046875


,child_id,path,label,logit0,logit1,prob0,prob1,prob1_cal
0,4,/content/drive/MyDrive/Stunted Children Identi...,1,-0.215767,0.020680,0.441162,0.558838,0.500469
1,13,/content/drive/MyDrive/Stunted Children Identi...,0,-0.054094,-0.275715,0.555179,0.444820,0.499560
2,24,/content/drive/MyDrive/Stunted Children Identi...,0,0.104901,0.140592,0.491078,0.508922,0.500071
3,28,/content/drive/MyDrive/Stunted Children Identi...,0,-0.119527,0.138631,0.435816,0.564184,0.500512
4,37,/content/drive/MyDrive/Stunted Children Identi...,0,-0.256898,-0.041548,0.446370,0.553630,0.500427


In [66]:
# Raw subject-level probabilities

THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 181), 3)
THRESHOLD_OBJECTIVE = "f2"

cal_curve, cal_thr = threshold_sweep(val_pred_cal, prob_col="prob1",
                                     thresholds=THRESHOLD_GRID, objective=THRESHOLD_OBJECTIVE)

print(f"Best Threshold on Validation: {cal_thr:0.4f}")
display(cal_curve[:5])

Best Threshold on Validation: 0.0500


,threshold,accuracy,precision,sensitivity,specificity,f1,f2,balanced_accuracy,TN,FP,FN,TP
0,0.050,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
1,0.055,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
2,0.060,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
3,0.065,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8
4,0.070,0.421053,0.421053,1.0,0.0,0.592593,0.784314,0.5,0,11,0,8


In [67]:
print('test evaluation:')
# Evaluation Example
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=cal_thr)
display(result)

test evaluation:


{'accuracy': 0.42105263157894735,
 'precision': 0.42105263157894735,
 'sensitivity': 1.0,
 'specificity': 0.0,
 'f1': 0.5925925925925926,
 'f2': 0.7843137254901961,
 'balanced_accuracy': 0.5,
 'TN': 0,
 'FP': 11,
 'FN': 0,
 'TP': 8}

In [68]:
# Confidence Interval Example
CI = bootstrap_subject_ci(subject_df=test_predict, prob_col="prob1", threshold=cal_thr, n_boot=2000, seed=SEED)
display(CI)

{'accuracy_mean': 42.27,
 'accuracy_std. dev': 11.42,
 'accuracy_ci_low': 21.05,
 'accuracy_ci_high': 63.16,
 'precision_mean': 42.27,
 'precision_std. dev': 11.42,
 'precision_ci_low': 21.05,
 'precision_ci_high': 63.16,
 'sensitivity_mean': 100.0,
 'sensitivity_std. dev': 0.0,
 'sensitivity_ci_low': 100.0,
 'sensitivity_ci_high': 100.0,
 'specificity_mean': 0.0,
 'specificity_std. dev': 0.0,
 'specificity_ci_low': 0.0,
 'specificity_ci_high': 0.0,
 'f1_mean': 58.5,
 'f1_std. dev': 11.48,
 'f1_ci_low': 34.78,
 'f1_ci_high': 77.42,
 'f2_mean': 77.11,
 'f2_std. dev': 8.73,
 'f2_ci_low': 57.14,
 'f2_ci_high': 89.55,
 'balanced_accuracy_mean': 50.0,
 'balanced_accuracy_std. dev': 0.0,
 'balanced_accuracy_ci_low': 50.0,
 'balanced_accuracy_ci_high': 50.0}

In [69]:
sc4b_reliability_result.append([
     "EfficientNet-B0", "Focal", "cal. tuned", np.round(cal_thr, 3),
     CI["accuracy_mean"], CI["accuracy_std. dev"],
     CI["sensitivity_mean"], CI["sensitivity_std. dev"],
     CI["specificity_mean"], CI["specificity_std. dev"],
     CI["f2_mean"], CI["f2_std. dev"],
     CI["sensitivity_ci_low"], CI["sensitivity_ci_high"],
     CI["specificity_ci_low"], CI["specificity_ci_high"],
     CI["f2_ci_low"], CI["f2_ci_high"],
])

# 5\. Result

In [70]:
sc4b_reliability_result_df = pd.DataFrame(sc4b_reliability_result,
                                          columns=['model', 'loss',
                                                   'dec.', 'tr.',
                                                   'acc-mean','acc-dev',
                                                   'sens-mean','sens-dev',
                                                   'spec-mean','spec-dev',
                                                   'f2-mean','f2-dev',
                                                   'sens-low','sens-high',
                                                   'spec-low','spec-high',
                                                   'f2-low','f2-high',
                                                   ])
display(sc4b_reliability_result_df)

base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/result/"
sc4b_reliability_result_df.to_excel(base + 'sc4b_reliability_result_df.xlsx', index=False)

,model,loss,dec.,tr.,acc-mean,acc-dev,sens-mean,sens-dev,spec-mean,spec-dev,f2-mean,f2-dev,sens-low,sens-high,spec-low,spec-high,f2-low,f2-high
0,EfficientNet-B0,CE,0.5,0.50,52.67,11.52,62.38,18.31,45.69,15.15,56.84,15.59,25.0,100.0,15.38,75.00,23.26,83.33
1,EfficientNet-B0,CE,tuned,0.05,42.27,11.42,100.00,0.00,0.00,0.00,77.11,8.73,100.0,100.0,0.00,0.00,57.14,89.55
2,EfficientNet-B0,CE,cal. tuned,0.05,42.27,11.42,100.00,0.00,0.00,0.00,77.11,8.73,100.0,100.0,0.00,0.00,57.14,89.55
3,EfficientNet-B0,Weighted CE,0.5,0.50,52.66,11.26,74.93,16.05,36.38,14.46,65.37,13.58,40.0,100.0,9.09,66.67,33.33,87.50
4,EfficientNet-B0,Weighted CE,tuned,0.05,42.27,11.42,100.00,0.00,0.00,0.00,77.11,8.73,100.0,100.0,0.00,0.00,57.14,89.55
5,EfficientNet-B0,Weighted CE,cal. tuned,0.05,42.27,11.42,100.00,0.00,0.00,0.00,77.11,8.73,100.0,100.0,0.00,0.00,57.14,89.55
6,EfficientNet-B0,Focal,0.5,0.50,36.83,10.73,74.69,15.41,9.15,8.81,60.97,12.71,40.0,100.0,0.00,30.00,32.61,81.97
7,EfficientNet-B0,Focal,tuned,0.05,42.27,11.42,100.00,0.00,0.00,0.00,77.11,8.73,100.0,100.0,0.00,0.00,57.14,89.55
8,EfficientNet-B0,Focal,cal. tuned,0.05,42.27,11.42,100.00,0.00,0.00,0.00,77.11,8.73,100.0,100.0,0.00,0.00,57.14,89.55


In [71]:
sc4b_loss_function_result_df = pd.DataFrame(sc4b_loss_function_result,
                                            columns=['model_name',
                                                     'scheme', 'model', 'optimizer', 'loss',
                                                     'evaluation',
                                                     'accuracy',
                                                     'precision',
                                                     'sensitivity',
                                                     'specificity',
                                                     'f1',
                                                     'f2',
                                                     'TN',
                                                     'FP',
                                                     'FN',
                                                     'TP',
                                                     ])
display(sc4b_loss_function_result_df)

base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/result/"
sc4b_loss_function_result_df.to_excel(base + 'sc4b_loss_function_result.xlsx', index=False)

,model_name,scheme,model,optimizer,loss,evaluation,accuracy,precision,sensitivity,specificity,f1,f2,TN,FP,FN,TP
0,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,CE,validation,0.578947,0.500000,0.500,0.636364,0.500000,0.500000,7,4,4,4
1,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,CE,test,0.526316,0.454545,0.625,0.454545,0.526316,0.581395,5,6,3,5
2,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,weighted-CE,validation,0.526316,0.454545,0.625,0.454545,0.526316,0.581395,5,6,3,5
3,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,weighted-CE,test,0.526316,0.461538,0.750,0.363636,0.571429,0.666667,4,7,2,6
4,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,focal,validation,0.526316,0.461538,0.750,0.363636,0.571429,0.666667,4,7,2,6
5,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,focal,test,0.368421,0.375000,0.750,0.090909,0.500000,0.625000,1,10,2,6
